In [2]:
import os
from dotenv import load_dotenv
load_dotenv()


True

1. from dotenv import load_dotenv: Imports load_dotenv from the python-dotenv package to read a local .env file.
2. load_dotenv(): Finds and loads the keys inside your .env file into the environment variables space.
3. os.getenv("GROQ_API_KEY"): Extracts your specific Groq API token so it can be passed to the LLM client securely without hardcoding it.
4. from langchain_core.messages import HumanMessage: Imports the specific object class representing text coming from an end-user.
5. model.invoke([...]): Sends a list containing a single user message to the LLM.
6. from langchain_core.messages import AIMessage: Imports the object class representing a response generated by the AI model.

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=os.getenv("GROQ_API_KEY"))

c:\Users\003X65744\GenAI\genaienv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, my name is Bikash and I am a functional consultant")])

AIMessage(content="Nice to meet you, Bikash. As a functional consultant, I'm guessing you work on implementing and configuring business software systems, such as ERP (Enterprise Resource Planning) or CRM (Customer Relationship Management) systems. You likely work closely with clients to understand their business needs and develop solutions to meet those needs.\n\nWhat specific area of functional consulting do you specialize in, Bikash? For example, are you focused on finance, supply chain, human capital management, or something else?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 48, 'total_tokens': 145, 'completion_time': 0.197157127, 'completion_tokens_details': None, 'prompt_time': 0.027139302, 'prompt_tokens_details': None, 'queue_time': 0.082601047, 'total_time': 0.224296429}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'mod

1. ChatMessageHistory: An in-memory utility class that stores lists of HumanMessage and AIMessage objects.
2. BaseChatMessageHistory: The base abstract type used for type hinting.
3. RunnableWithMessageHistory: A helper class that dynamically wraps any execution chain/model to automatically manage pulling and pushing interaction history.
4. store={}: A simple dictionary acting as an in-memory database to keep track of different user sessions.
5. Session Management Logic: A helper function that takes a session_id string. If that ID doesn't exist in our dictionary store, it provisions a fresh instance of ChatMessageHistory. It then returns that user's specific history log.
6. RunnableWithMessageHistory(model, get_session_history): Wraps the base model inside the history manager. Whenever .invoke() is run on with_message_history, it calls get_session_history in the background.
7. config=...: Defines configuration dictionaries that pass metadata parameters (like specific session_id codes) to execution runs.
8. Execution: Sends the message under the context of "chat1". The history helper captures the user's input and the model's generated response, automatically saving both into store["chat1"].

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, my name is Bikash and I am a functional consultant"),
        AIMessage(content="Nice to meet you, Bikash. As a functional consultant, I'm sure you work with various systems and software to help organizations improve their processes and operations. Which specific areas or industries do you typically work with, and what's your favorite project that you've worked on recently?\n"),
        HumanMessage(content="Hey what's my name and what do I do?")
        
    ]
)

AIMessage(content="Your name is Bikash, and you're a functional consultant.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 125, 'total_tokens': 139, 'completion_time': 0.015371655, 'completion_tokens_details': None, 'prompt_time': 0.007115954, 'prompt_tokens_details': None, 'queue_time': 0.15831852, 'total_time': 0.022487609}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e69c7-8bcc-7890-8ea4-76c06396e8f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 125, 'output_tokens': 14, 'total_tokens': 139})

In [7]:
### Message History
# We can use this class to wrap our model and make it stateful. This will keep a track of outputs of the model.

In [8]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

C:\Users\003X65744\AppData\Local\Temp\ipykernel_18804\40209978.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\003X65744\GenAI\genaienv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [9]:
config={"configurable":{"session_id":"chat1"}}

In [10]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, My favorite food is Paneer and I love it with Maggi masala.")], 
    config=config
)

In [11]:
response.content

"That's an interesting combination. Paneer is a popular Indian cheese, and Maggi masala is a popular instant noodle brand in India. While they may not be a traditional pairing, many people enjoy the creamy texture of paneer with the spicy and savory flavor of Maggi masala.\n\nIf you're looking to elevate your Paneer-Maggi masala dish, here are a few suggestions:\n\n1. Add some aromatics: Saute onions, garlic, and ginger before adding the Maggi masala and paneer for added depth of flavor.\n2. Spice it up: Add some red chili flakes or sriracha to give your dish an extra kick.\n3. Mix and match: Try pairing paneer with other instant noodle flavors, like Maggi vegetable or Maggi chili.\n4. Get creative: Add some vegetables like bell peppers, carrots, or broccoli to make your dish more nutritious and interesting.\n\nHow do you typically prepare your Paneer-Maggi masala dish? Do you have any favorite variations or tips to share?"

In [12]:
## Changing the config


config1={"configurable":{"session_id":"chat1"}}
response = with_message_history.invoke(
    [HumanMessage(content="What's my favorite food ?")], 
    config=config1
)

In [13]:
response.content

'Your favorite food is Paneer, and you love it with Maggi masala.'

1. ChatPromptTemplate.from_messages: Pre-formats a structural prompt sequence.
2. ("system", "..."): Injecting static System behavior guidelines to control the model's overall persona.
3. MessagesPlaceholder(variable_name="messages"): Generates a dynamic structural zone inside the prompt array where an existing list of chat messages will be dropped in seamlessly.
4. chain = prompt | model: Uses the LCEL Pipe Operator (|) to link components together. The output dictionary from the prompt template flows straight into the input of the LLM model.
5. Multi-variable Complex Prompts: Introducing a standard variable token string {language} alongside the chat history list variable messages.
6. input_messages_key="messages": Because our new dictionary input payload contains both a language key and a messages key, we must explicitly instruct RunnableWithMessageHistory which structural dictionary key contains the target list intended for chat historical tracking.

In [14]:
## Prompt Templates

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant and answer all the question with the best of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages")

    ]
)

chain = prompt|model 

In [15]:
chain.invoke({"messages":[HumanMessage(content="Hi, I love paneer")], "language":"Hindi"})

AIMessage(content='पनीर पसंदीदा है! (Paneer pasandeeda hai!) आप कौन सा पनीर व्यंजन पसंद करते हैं? क्या आप पनीर टिक्का मसाला, पनीर मटर, या कोई अन्य व्यंजन पसंद करते हैं?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 59, 'total_tokens': 133, 'completion_time': 0.195988762, 'completion_tokens_details': None, 'prompt_time': 0.00294609, 'prompt_tokens_details': None, 'queue_time': 0.053936134, 'total_time': 0.198934852}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e69c7-ca8a-7a32-b8a6-5e2b59841bef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 74, 'total_tokens': 133})

In [16]:
with_message_history= RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

c:\Users\003X65744\GenAI\genaienv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [17]:
config = {"configurable": {"session_id":"chat4"}}
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="I love paneer")], "language":"Hindi"}, 
    config=config
)

In [18]:
response.content

'आपको पनीर पसंद है! पनीर एक बहुत ही लोकप्रिय और स्वादिष्ट भारतीय डिश है, जो मुख्य रूप से दही से बनाया जाता है। यह विभिन्न प्रकार के व्यंजनों में उपयोग किया जाता है, जैसे कि करी, टिक्का, कढ़ाई, और बहुत कुछ। पनीर के कई स्वादिष्ट प्रकार होते हैं, जैसे कि पनीर क्रीम, पनीर टिक्का, पनीर मटर, और पनीर कढ़ाई। आपको कौन सा पनीर व्यंजन सबसे ज्यादा पसंद है?'

In [19]:
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="which recipe is the best ?")], "language":"Hindi"}, 
    config=config
)
response.content

'पनीर के विभिन्न व्यंजन हैं और प्रत्येक का अपना एक अनोखा स्वाद है। यहाँ कुछ लोकप्रिय पनीर व्यंजन हैं जिन्हें आप आजमा सकते हैं:\n\n1. **पनीर टिक्का**: यह एक स्वादिष्ट और लोकप्रिय भारतीय व्यंजन है, जिसमें पनीर को मसालों और दही में मैरीनेट किया जाता है और फिर ग्रिल किया जाता है।\n2. **पनीर मटर**: यह एक स्वादिष्ट और स्वस्थ व्यंजन है, जिसमें पनीर और मटर को मसालों और दही में पकाया जाता है।\n3. **पनीर कढ़ाई**: यह एक स्वादिष्ट और मसालेदार व्यंजन है, जिसमें पनीर को मसालों और दही में पकाया जाता है।\n4. **पनीर पुलाव**: यह एक स्वादिष्ट और आसानी से बनाने योग्य व्यंजन है, जिसमें पनीर, चावल, और मसालों को एक साथ पकाया जाता है।\n\nइनमें से कौन सा व्यंजन आपको सबसे ज्यादा पसंद है?'

In [20]:
## Manage the Conversation History

1. trim_messages(...): Builds a specialized preprocessing transformer that inspects message histories.
2. max_tokens=70: Keeps the input window aggressively small (capped at 70 tokens max).
3. strategy="last": Keeps the newest messages and cuts out older ones if things overflow.
4. token_counter=model: Instructs the tool to utilize the model's own built-in internal tokenizer to calculate the true length count.
5. include_system=True: Guarantees that the underlying system persona prompt message never gets discarded.
6. allow_partial=False: Prevents split-token slicing; a whole message must fit, or it gets skipped entirely.
7. start_on="human": Requires the sliced conversational thread slice window to start with a user message.
8. itemgetter("messages") | trimmer: Fetches the message array out of the incoming input payload dictionary data structure and runs it straight through the trimmer.
9. RunnablePassthrough.assign(messages=...): Updates the dynamic input payload metadata structure, replacing the raw messages entry with the newly truncated shorter list, before sending everything to the prompt layout step.

In [27]:
from langchain_core.messages import SystemMessage, trim_messages
## trim messages - helps to reduce the how many messages we are sending to the model. The trimmers allows us to specify how many tokens we
# want to keep , along with parameters like if we want to always keep the system message and whether to allow partial messages.

trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages=[
    SystemMessage(content="You are a good assistant"),
    HumanMessage(content="Hi , I am Bikash"),
    AIMessage(content="Hi, how are you!"),
    HumanMessage(content="i'm fine, how are you"),
    AIMessage(content="I am also fine. Do you want to learn something today?"),
    HumanMessage(content="Can you tell me the capital of India ?"),
    AIMessage(content="Sure, the capital of India is New Delhi. Anything else do you want to know about New Delhi ?"),
    HumanMessage(content="How is the AQI in Delhi ?"),
    AIMessage(content="the AQI of New Delhi is currently 300 AQI"),
    HumanMessage(content="Ohh, that's too much, right ?"),
    AIMessage(content="Yes, you should avoid getting exposed to the outside air and use Air purifiers if it's possible.")
]

trimmer.invoke(messages)

[SystemMessage(content='You are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='How is the AQI in Delhi ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='the AQI of New Delhi is currently 300 AQI', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="Ohh, that's too much, right ?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="Yes, you should avoid getting exposed to the outside air and use Air purifiers if it's possible.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [28]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages = itemgetter("messages")|trimmer)
    |prompt
    |model
)

response = chain.invoke(
    {
    "messages":messages + [HumanMessage(content="what suggestion did you give me to prevent from bad AQI?")],
    "language":"English"
    }
)

response.content

'To prevent exposure to bad Air Quality Index (AQI), I suggested using an air purifier if possible.'

In [29]:
## Wrap this in Message History

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config={"configurable":{"session_id":"chat5"}}

c:\Users\003X65744\GenAI\genaienv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [32]:
response = with_message_history.invoke(
    {
        "messages":messages + [HumanMessage(content="what suggestion did you give me to prevent from bad AQI?")],
        "language":"English"
    }, 
    config=config
)
response.content

'I made a mistake earlier. You didn\'t ask about Air Quality Index (AQI) or bad air quality. Your original statement "Ohh, that\'s too much, right?" is a bit ambiguous, but I\'m going to take a guess that you might be expressing surprise or disagreement about something. Could you please provide more context or clarify what you meant by that statement?'